# Exploratory Data Analysis — Solar Flares Dataset

**Prerequisite (pre-Module 1):** Foundational data handling with NumPy, Pandas, and Matplotlib.

This notebook covers:
1. Loading the dataset (via `kagglehub`)
2. Data cleaning (handling missing values)
3. Summary statistics
4. Distribution plots
5. Correlation analysis


## 1. Imports

In [ ]:
import kagglehub
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Nicer default plot style
plt.style.use('seaborn-v0_8-darkgrid')
pd.set_option('display.max_columns', 50)


## 2. Download and Load the Dataset

In [ ]:
# Download latest version of the dataset from Kaggle
path = kagglehub.dataset_download("stealthtechnologies/solar-flares-dataset")
print("Path to dataset files:", path)

# List files in the downloaded folder
files = os.listdir(path)
print("Files found:", files)


In [ ]:
# Load the first CSV file found in the dataset folder
csv_files = [f for f in files if f.endswith('.csv')]
print("CSV files:", csv_files)

file_path = os.path.join(path, csv_files[0])
df = pd.read_csv(file_path)

print("Shape:", df.shape)
df.head()


## 3. Initial Data Overview

In [ ]:
df.info()


In [ ]:
print("Number of duplicate rows:", df.duplicated().sum())
print("\nColumn data types:\n", df.dtypes)


## 4. Handling Missing Values

In [ ]:
# Count missing values per column
missing = df.isnull().sum()
missing_pct = (missing / len(df)) * 100

missing_df = pd.DataFrame({'missing_count': missing, 'missing_pct': missing_pct})
missing_df = missing_df[missing_df['missing_count'] > 0].sort_values('missing_pct', ascending=False)
missing_df


In [ ]:
# Visualize missing values
if len(missing_df) > 0:
    plt.figure(figsize=(10, 5))
    plt.bar(missing_df.index.astype(str), missing_df['missing_pct'])
    plt.xticks(rotation=90)
    plt.ylabel("% Missing")
    plt.title("Percentage of Missing Values per Column")
    plt.tight_layout()
    plt.show()
else:
    print("No missing values found in the dataset.")


In [ ]:
# Cleaning strategy:
# - Drop columns with more than 50% missing values (too sparse to be useful)
# - Fill remaining numeric missing values with the column median
# - Fill remaining categorical/object missing values with the mode

threshold = 0.5
cols_to_drop = missing_df[missing_df['missing_pct'] > threshold * 100].index.tolist()
df_clean = df.drop(columns=cols_to_drop)
print("Dropped columns (>50% missing):", cols_to_drop)

numeric_cols = df_clean.select_dtypes(include=[np.number]).columns
categorical_cols = df_clean.select_dtypes(exclude=[np.number]).columns

for col in numeric_cols:
    if df_clean[col].isnull().any():
        df_clean[col] = df_clean[col].fillna(df_clean[col].median())

for col in categorical_cols:
    if df_clean[col].isnull().any():
        mode_val = df_clean[col].mode()
        if not mode_val.empty:
            df_clean[col] = df_clean[col].fillna(mode_val[0])

print("\nRemaining missing values after cleaning:", df_clean.isnull().sum().sum())
df_clean.head()


## 5. Summary Statistics

In [ ]:
df_clean.describe().T


In [ ]:
# Summary stats for categorical columns (if any remain)
if len(categorical_cols) > 0:
    display(df_clean[categorical_cols].describe().T)
else:
    print("No categorical columns present.")


## 6. Distribution Plots

In [ ]:
# Plot histograms for all numeric columns
num_cols = df_clean.select_dtypes(include=[np.number]).columns
n_cols = 3
n_rows = int(np.ceil(len(num_cols) / n_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 4 * n_rows))
axes = np.array(axes).flatten()

for i, col in enumerate(num_cols):
    axes[i].hist(df_clean[col].dropna(), bins=30, color='darkorange', edgecolor='black')
    axes[i].set_title(col)
    axes[i].set_xlabel(col)
    axes[i].set_ylabel("Frequency")

# Hide any unused subplots
for j in range(i + 1, len(axes)):
    axes[j].axis('off')

plt.tight_layout()
plt.show()


In [ ]:
# Boxplots to inspect outliers for numeric columns
fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 4 * n_rows))
axes = np.array(axes).flatten()

for i, col in enumerate(num_cols):
    axes[i].boxplot(df_clean[col].dropna(), vert=False)
    axes[i].set_title(col)

for j in range(i + 1, len(axes)):
    axes[j].axis('off')

plt.tight_layout()
plt.show()


In [ ]:
# If a categorical target/class column exists (e.g. flare class), show its distribution
if len(categorical_cols) > 0:
    for col in categorical_cols:
        plt.figure(figsize=(8, 4))
        df_clean[col].value_counts().plot(kind='bar', color='firebrick', edgecolor='black')
        plt.title(f"Distribution of {col}")
        plt.ylabel("Count")
        plt.xlabel(col)
        plt.tight_layout()
        plt.show()


## 7. Correlation Analysis

In [ ]:
corr = df_clean[num_cols].corr()

plt.figure(figsize=(10, 8))
im = plt.imshow(corr, cmap='coolwarm', vmin=-1, vmax=1)
plt.colorbar(im, label='Correlation coefficient')
plt.xticks(range(len(corr.columns)), corr.columns, rotation=90)
plt.yticks(range(len(corr.columns)), corr.columns)
plt.title("Correlation Matrix (Numeric Features)")
plt.tight_layout()
plt.show()

corr


In [ ]:
# Top absolute correlations (excluding self-correlation)
corr_pairs = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool)).stack()
top_corr = corr_pairs.abs().sort_values(ascending=False).head(10)
print("Top 10 strongest correlations:")
top_corr


## 8. Conclusion

- Loaded the dataset directly from Kaggle using `kagglehub`.
- Identified and handled missing values (dropped very sparse columns, imputed the rest with median/mode).
- Generated summary statistics for numeric and categorical columns.
- Visualized distributions (histograms, boxplots, and class-count bar charts) to understand spread, outliers, and class balance.
- Computed and visualized the correlation matrix to identify relationships between numeric features.

This forms the foundational data-handling workflow to be reused in later experiments/modules.
